# Notebook 11 — Final Analysis and Comprehensive Report

**Objective:** Consolidate all results, compare SOH and RUL models,
generate publication-quality plots, and summarise project findings.

## Summary of Results (from run_ml_pipeline.py)

### SOH Results
| Rank | Model | Test RMSE (%) | Test R² |
|------|-------|--------------|---------|
| 1 | Gradient Boosting (tuned) | 8.42 | 0.857 |
| 2 | Linear Regression | 8.73 | 0.846 |
| 3 | Gradient Boosting | 9.12 | 0.832 |
| 4 | Random Forest | 10.76 | 0.767 |
| 5 | Random Forest (tuned) | 11.19 | 0.748 |

### RUL Results
| Rank | Model | Test RMSE (cycles) | Test R² |
|------|-------|-------------------|---------|
| 1 | Random Forest (tuned) | 6.99 | 0.573 |
| 2 | Gradient Boosting | 8.83 | 0.321 |
| 3 | Gradient Boosting (tuned) | 9.97 | 0.134 |
| 4 | Random Forest | 10.08 | 0.114 |
| 5 | Linear Regression | 18.55 | −2.000 |

---

**Key Findings:**
- SOH prediction is reliable (R²=0.857): discharge features strongly encode battery health
- RUL prediction is harder (R²=0.573): requires knowledge of degradation trajectory, not just current state
- Non-linear models outperform Linear Regression for RUL; the opposite holds for SOH due to data heterogeneity
- 11 of 34 batteries never reached 70% EOL (too lightly degraded in the test duration)

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib

ROOT = os.path.dirname(os.path.abspath(''))
sys.path.insert(0, os.path.join(ROOT, 'src'))

from evaluation import compute_metrics, build_comparison_table, print_metrics_table

sns.set_style('whitegrid')
plt.rcParams.update({'font.size': 11})

## 1. Load All Results

In [ ]:
final  = pd.read_csv(os.path.join(ROOT, 'cleaned_data', 'final_dataset.csv'))
soh_m  = pd.read_csv(os.path.join(ROOT, 'results', 'metrics', 'soh_metrics.csv'), index_col=0)
rul_m  = pd.read_csv(os.path.join(ROOT, 'results', 'metrics', 'rul_metrics.csv'), index_col=0)
soh_p  = pd.read_csv(os.path.join(ROOT, 'results', 'predictions', 'soh_predictions.csv'))
rul_p  = pd.read_csv(os.path.join(ROOT, 'results', 'predictions', 'rul_predictions.csv'))

best_soh = joblib.load(os.path.join(ROOT, 'models', 'soh', 'best_soh_model.pkl'))
best_rul = joblib.load(os.path.join(ROOT, 'models', 'rul', 'best_rul_model.pkl'))

print('SOH Metrics:')
print(soh_m[['model', 'RMSE_test', 'MAE_test', 'R2_test']].to_string())
print('\nRUL Metrics:')
print(rul_m[['model', 'RMSE_test', 'MAE_test', 'R2_test']].to_string())

## 2. SOH Degradation Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: all batteries
ax = axes[0]
for bid in sorted(final['battery_id'].unique()):
    grp = final[final['battery_id'] == bid].sort_values('cycle')
    ax.plot(grp['cycle'], grp['soh'], alpha=0.55, linewidth=1)
ax.axhline(70, color='red', linestyle='--', linewidth=2, label='EOL (70%)')
ax.set_xlabel('Discharge Cycle')
ax.set_ylabel('SOH (%)')
ax.set_title('SOH Degradation — All 34 Batteries')
ax.legend()

# Right: actual vs predicted
ax = axes[1]
ax.scatter(soh_p['soh'], soh_p['soh_predicted'],
           alpha=0.4, s=18, color='steelblue', edgecolors='k', linewidths=0.2)
lim = [max(0, soh_p['soh'].min()-5), soh_p['soh'].max()+5]
ax.plot(lim, lim, 'r--', linewidth=1.5, label='Perfect fit')
m = compute_metrics(soh_p['soh'], soh_p['soh_predicted'])
ax.set_xlabel('Actual SOH (%)')
ax.set_ylabel('Predicted SOH (%)')
ax.set_title(f'Best SOH Model — Actual vs Predicted\n(RMSE={m["RMSE"]:.3f}%  R²={m["R2"]:.3f})')
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'plots', 'soh', 'nb11_soh_summary.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## 3. RUL Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# RUL actual vs predicted
ax = axes[0]
ax.scatter(rul_p['rul'], rul_p['rul_predicted'],
           alpha=0.4, s=18, color='#C62828', edgecolors='k', linewidths=0.2)
lim = [max(0, rul_p['rul'].min()-2), rul_p['rul'].max()+5]
ax.plot(lim, lim, 'b--', linewidth=1.5, label='Perfect fit')
m2 = compute_metrics(rul_p['rul'], rul_p['rul_predicted'])
ax.set_xlabel('Actual RUL (cycles)')
ax.set_ylabel('Predicted RUL (cycles)')
ax.set_title(f'Best RUL Model — Actual vs Predicted\n(RMSE={m2["RMSE"]:.3f} cy  R²={m2["R2"]:.3f})')
ax.legend()

# RUL residuals
ax = axes[1]
ax.scatter(rul_p['rul_predicted'], rul_p['residual'],
           alpha=0.4, s=18, color='#C62828', edgecolors='k', linewidths=0.2)
ax.axhline(0, color='black', linestyle='--', linewidth=1.5)
ax.set_xlabel('Predicted RUL (cycles)')
ax.set_ylabel('Residual (Actual − Predicted)')
ax.set_title('RUL Residual Plot')

plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'plots', 'rul', 'nb11_rul_summary.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## 4. Model Comparison Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# SOH RMSE comparison
ax = axes[0]
soh_sorted = soh_m.sort_values('RMSE_test')
colors = ['#2E7D32' if i == 0 else '#78909C' for i in range(len(soh_sorted))]
bars = ax.bar(soh_sorted['model'], soh_sorted['RMSE_test'], color=colors)
ax.set_ylabel('Test RMSE (%)')
ax.set_title('SOH — Test RMSE by Model')
ax.set_xticklabels(soh_sorted['model'], rotation=30, ha='right', fontsize=9)
for bar, v in zip(bars, soh_sorted['RMSE_test']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{v:.3f}', ha='center', va='bottom', fontsize=8)

# RUL RMSE comparison
ax = axes[1]
rul_sorted = rul_m.sort_values('RMSE_test')
colors = ['#2E7D32' if i == 0 else '#78909C' for i in range(len(rul_sorted))]
bars = ax.bar(rul_sorted['model'], rul_sorted['RMSE_test'], color=colors)
ax.set_ylabel('Test RMSE (cycles)')
ax.set_title('RUL — Test RMSE by Model')
ax.set_xticklabels(rul_sorted['model'], rotation=30, ha='right', fontsize=9)
for bar, v in zip(bars, rul_sorted['RMSE_test']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{v:.2f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'plots', 'soh', 'nb11_model_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## 5. Dataset & Battery Summary

In [ ]:
# Overall dataset stats
print('='*60)
print('  DATASET SUMMARY')
print('='*60)
print(f'  Total discharge cycles    : {len(final)}')
print(f'  Total batteries           : {final["battery_id"].nunique()}')
print(f'  Cycles per battery        : {final.groupby("battery_id").size().describe().to_dict()}')
print()

# Battery grouping
rul_df  = final.dropna(subset=['rul'])
eol_bats   = sorted(rul_df['battery_id'].unique())
non_eol    = sorted(set(final['battery_id'].unique()) - set(eol_bats))
print(f'  EOL batteries (in RUL)   : {len(eol_bats)}  {eol_bats}')
print(f'  Non-EOL (excluded RUL)   : {len(non_eol)}   {non_eol}')
print()

# SOH stats
print(f'  SOH range overall         : {final["soh"].min():.1f}% – {final["soh"].max():.1f}%')
print(f'  SOH mean                  : {final["soh"].mean():.1f}%  ± {final["soh"].std():.1f}%')
print()
print(f'  RUL range (EOL bats only) : {rul_df["rul"].min():.0f} – {rul_df["rul"].max():.0f} cycles')
print(f'  RUL mean                  : {rul_df["rul"].mean():.1f} cycles')

## 6. Saved Files Report

In [ ]:
import glob

print('SAVED OUTPUT FILES')
print('='*55)
for pattern in [
    'cleaned_data/*.csv',
    'models/soh/*.pkl', 'models/rul/*.pkl',
    'results/metrics/*.csv', 'results/predictions/*.csv',
    'results/tables/*.csv',
    'plots/soh/*.png', 'plots/rul/*.png',
]:
    files = sorted(glob.glob(os.path.join(ROOT, pattern)))
    if files:
        print(f'\n  {pattern}:')
        for f in files:
            size_kb = os.path.getsize(f) / 1024
            print(f'    {os.path.basename(f):<40} {size_kb:6.1f} KB')

## 7. Conclusions

### SOH Prediction
- **Best model:** Gradient Boosting (tuned) — Test RMSE=8.42%, R²=0.857
- `energy_Wh`, `voltage_mean`, and `cycle` are the top predictors
- The model generalises well to unseen batteries (10 test batteries)
- Non-monotonic batteries (mixed protocols) add noise; excluding them would likely improve RMSE further

### RUL Prediction
- **Best model:** Random Forest (tuned) — Test RMSE=6.99 cycles, R²=0.573
- RUL prediction is harder because:
  - Some test batteries already reached EOL (RUL=0 for most cycles)
  - Single-cycle features do not capture the degradation *trajectory*
  - NASA dataset uses varied discharge protocols — models trained on 4A protocols struggle to generalise to 1A test batteries
- Future improvement: Add **lag features** (SOH slope over last N cycles) and **battery protocol flag**

### Limitations
- 11 batteries excluded from RUL training (never reached 70% SOH)
- Batteries with mixed discharge protocols show non-monotonic SOH curves
- Dataset size (2,737 cycles, 34 batteries) is moderate for deep learning approaches
- No LSTM or time-series models were used (intentional: scikit-learn based pipeline)